In [991]:
import pandas as pd
print(pd.__version__)

3.1.1


In [992]:
from pathlib import Path
import io

RAW = Path("../data_market/raw")
PROCESSED = Path("../data_market/processed")
PROCESSED.mkdir(exist_ok=True)
INIZIO= "2016"

list(RAW.iterdir())    #iterdir()--> generatore che ti dice cosa sta in una cartella/sottocartella/file

[WindowsPath('../data_market/raw/bce_eur_usd.csv'),
 WindowsPath('../data_market/raw/eventi.csv'),
 WindowsPath('../data_market/raw/fao_food_price_index.csv'),
 WindowsPath('../data_market/raw/gpr_index.xls'),
 WindowsPath('../data_market/raw/istat_coefficienti_raccordo.xlsx'),
 WindowsPath('../data_market/raw/istat_nic_2015_2025.csv'),
 WindowsPath('../data_market/raw/istat_nic_2026.csv'),
 WindowsPath('../data_market/raw/worldbank_pinksheet.xlsx')]

In [993]:
COL_DATA = "TIME_PERIOD"
COL_VALORE = "Osservazione"
COL_PRODOTTO = "ECOICOP 2"
COL_CODICE = "ECOICOP_2"

PRODOTTI = {
    "011131": "pane",
    "01115": "pasta",
    "011511": "olio_girasole",
    "012201": "caffe",
    "01181": "zucchero",
    "01141": "latte",
    "0722": "carburanti",
}

In [994]:
def istat_largo(df):
    """Tiene i 7 prodotti e trasforma la tabella ISTAT da formato lungo a largo."""
    sel = df[df[COL_CODICE].isin(PRODOTTI.keys())]  #df[COL_CODICE].isin(PRODOTTI.keys())--->restituisce valori booleani che fanno da filtro
    largo = sel.pivot(index=COL_DATA, columns=COL_CODICE, values=COL_VALORE)  #righe->data;colonne->ECOICOP_2;Valori della tabella->Osservazioni
    largo = largo.rename(columns=PRODOTTI)
    largo.index = pd.to_datetime(largo.index) #trasforma le righe da string a data vera
    largo.index.name = "data" #da il nome "data" alle righe
    largo.columns.name = None
    return largo

In [995]:
istat_old = pd.read_csv(
    RAW / "istat_nic_2015_2025.csv",
    quotechar="'",               # i testi sono racchiusi tra apici singoli
    dtype={"ECOICOP_2": str},  # perchè ci sono codiceìi con lo 0 davanti e se fosse visto come intero verrebbe cancellato
)
istat_old.head()

,FREQ,Frequenza,REF_AREA,Territorio,DATA_TYPE,Indicatore,MEASURE,Misura,ECOICOP_2,ECOICOP 2,TIME_PERIOD,Osservazione,OBS_STATUS,"Stato dell""osservazione'"
0,M,Mensile,IT,Italia,39,"Indice dei prezzi al consumo per l""intera coll...",4,Numeri indici,00,Indice generale,2016-01,99.6,NaN,NaN
1,M,Mensile,IT,Italia,39,"Indice dei prezzi al consumo per l""intera coll...",4,Numeri indici,00,Indice generale,2016-02,99.4,NaN,NaN
2,M,Mensile,IT,Italia,39,"Indice dei prezzi al consumo per l""intera coll...",4,Numeri indici,00,Indice generale,2016-03,99.6,NaN,NaN
3,M,Mensile,IT,Italia,39,"Indice dei prezzi al consumo per l""intera coll...",4,Numeri indici,00,Indice generale,2016-04,99.5,NaN,NaN
4,M,Mensile,IT,Italia,39,"Indice dei prezzi al consumo per l""intera coll...",4,Numeri indici,00,Indice generale,2016-05,99.8,NaN,NaN


In [996]:
print(istat_old.shape)
istat_old.iloc[:, 9].unique()   # la decima colonna: i nomi dei prodotti

(93600, 14)


<StringArray>
[                                        'Indice generale',
               'Prodotti alimentari e bevande analcoliche',
                                     'Prodotti Alimentari',
                    'Cereali e prodotti a base di cereali',
                                                 'Cereali',
                                                'Frumento',
                                                    'Riso',
                                                    'Orzo',
                                           'Altri Cereali',
                                       'Farina di cereali',
 ...
                               'Servizi non regolamentati',
                         '-- beni e servizi regolamentati',
                                      'Beni regolamentati',
                                   'Servizi regolamentati',
                           '-- alta frequenza d"acquisto'',
                          '-- bassa frequenza d"acquisto'',
                     

In [997]:
prodotti = pd.Series(istat_old[COL_PRODOTTO].unique())

parole = ["pane", "maccheroni", "olio", "oli ", "caff", "zucchero", "latte", "benzina", "gasolio", "carburant"]
for p in parole:
    trovati = prodotti[prodotti.str.contains(p, case=False, na=False)]
    print(f"--- {p} ---")
    print(trovati.tolist())

--- pane ---
['Pane e prodotti da forno', 'Pane']
--- maccheroni ---
['Maccheroni, tagliatelle, cuscus e paste alimentari simili']
--- olio ---
['Olio di Semi Girasole', 'Olio di oliva', 'Olio di Semi Arachide', 'Olio di Semi Mais', 'Gasolio per riscaldamento', 'Gasolio per mezzi di trasporto']
--- oli  ---
['Oli e grassi', 'Oli vegetali', 'Altri oli vegetali', 'Altri articoli di abbigliamento e accessori per l"abbigliamento\'', 'Altri articoli di abbigliamento', 'Altri articoli d"abbigliamento\'', 'Articoli per l"illuminazione\'', 'Piccoli elettrodomestici', 'Piccoli apparecchi per la cottura e la lavorazione degli alimenti', 'Piccoli apparecchi per la preparazione di bevande', 'Altri piccoli elettrodomestici', 'Utensili e articoli da cucina', 'Utensili da cucina non elettrici e articoli domestici non elettrici', 'Piccoli utensili', 'Piccoli accessori elettrici', 'Beni non durevoli per l"abitazione\'', 'Altri beni non durevoli per l"abitazione\'', 'Servizi di lavanderia per articoli t

## Selezione dei prodotti

Dal paniere ISTAT (646 voci) teniamo 7 prodotti legati a materie prime
importate o a rotte commerciali esposte alle crisi geopolitiche.
I prodotti sono selezionati per **codice ECOICOP 2**, non per nome,
perché alcuni nomi compaiono più volte nel file (ad esempio lo zucchero
appare sia come sottoclasse `01181` sia come segmento `011811`).

| Codice ECOICOP 2 | Voce ISTAT | Nome nel dataset | Perché è nel carrello (IPOTESI) |
|---|---|---|---|
| `011131` | Pane | `pane` | Grano (Ucraina e Russia tra i grandi esportatori) |
| `01115` | Maccheroni, tagliatelle, cuscus e paste alimentari simili | `pasta` | Grano duro |
| `011511` | Olio di semi di girasole | `olio_girasole` | L'Ucraina è il primo esportatore mondiale di girasole |
| `012201` | Caffè | `caffe` | Importato, sensibile a cambio e trasporti |
| `01181` | Zucchero di canna e zucchero di barbabietola | `zucchero` | Materia prima agricola scambiata a livello mondiale |
| `01141` | Latte crudo e intero | `latte` | Sensibile ai costi di energia e mangimi |
| `0722` | Carburanti e lubrificanti per mezzi personali di trasporto | `carburanti` | Legato direttamente al prezzo del petrolio |

I codici hanno lunghezze diverse (4, 5 o 6 cifre) perché ogni prodotto è
preso al livello di dettaglio in cui ISTAT lo identifica in modo specifico.

In [998]:
old = istat_old[istat_old[COL_CODICE].isin(PRODOTTI.keys())]    #isin() lascia solo le righe con quel valore della chiave del dizionario PRODOTTO
old.groupby(COL_CODICE).size()


ECOICOP_2
011131    120
01115     120
01141     120
011511    120
01181     120
012201    120
0722      120
dtype: int64


### Come leggere i valori

I valori **non sono prezzi in euro**: sono gli **indici NIC** di ciascun
prodotto (Indice nazionale dei prezzi al consumo per l'intera collettività).
In questo file sono espressi in **base 2015 = 100**, cioè 100 è il prezzo
medio del prodotto nel 2015.

**Valore − 100 = variazione percentuale rispetto alla media del 2015**

| Esempio | Valore | Significato |
|---|---|---|
| Pasta, gennaio 2016 | 100,2 | costava lo 0,2% in più rispetto al 2015 |
| Carburanti, febbraio 2016 | 89,3 | costavano il 10,7% in meno rispetto al 2015 |
| Zucchero, maggio 2016 | 97,7 | costava il 2,3% in meno rispetto al 2015 |

Questi sono indici dei **singoli prodotti**, non il NIC generale (che è la
media pesata di tutte le 646 voci del paniere).

> Nel passo successivo tutte le serie vengono convertite in **base 2025 = 100**
> con i coefficienti di raccordo ISTAT, per poterle unire ai dati del 2026.

In [999]:
old_largo = old.pivot(index=COL_DATA, columns=COL_CODICE, values=COL_VALORE)
old_largo = old_largo.rename(columns=PRODOTTI)
old_largo.index = pd.to_datetime(old_largo.index)   # "2016-01" diventa una data vera
old_largo.index.name = "data"
old_largo.columns.name = None                        # toglie l'etichetta "ECOICOP_2"
old_largo.head()

,pane,pasta,latte,olio_girasole,zucchero,caffe,carburanti
data,,,,,,,
2016-01-01,100.5,100.2,100.0,99.9,98.4,100.1,91.6
2016-02-01,100.6,100.1,99.9,99.7,98.2,100.1,89.3
2016-03-01,100.6,100.1,100.0,99.8,98.2,99.8,90.1
2016-04-01,100.7,99.9,99.7,99.5,98.2,99.8,91.6
2016-05-01,100.7,99.7,99.5,99.2,97.7,99.6,93.4


In [1000]:
def leggi_istat(percorso):
    with open(percorso, encoding="utf-8") as f:
        testo = f.read().replace("\"'", "’")   # "' diventa ’
    return pd.read_csv(io.StringIO(testo), quotechar="'", dtype={"ECOICOP_2": str})  #io.StringIO() legge istat_nic_2026 che ho fatto diventare la riga prima un file di testo, come un csv

istat_new = leggi_istat(RAW / "istat_nic_2026.csv")
print(istat_new.shape)
print(istat_new.columns.tolist())

(8012, 14)
['FREQ', 'Frequenza', 'REF_AREA', 'Territorio', 'DATA_TYPE', 'Indicatore', 'MEASURE', 'Misura', 'ECOICOP_2', 'ECOICOP 2', 'TIME_PERIOD', 'Osservazione', 'OBS_STATUS', 'Stato dell’osservazione']


L'ultimo mese disponibile è **provvisorio** (`OBS_STATUS = p`):
ISTAT lo pubblica come stima e lo conferma o corregge il mese successivo.

In [1001]:
istat_new["OBS_STATUS"].value_counts(dropna=False)
istat_new[istat_new["OBS_STATUS"] == "p"]["TIME_PERIOD"].unique()

<StringArray>
['2026-10']
Length: 1, dtype: str

Nella stima provvisoria di ottobre 2026 mancano i prodotti a 6 cifre
(pane, olio di girasole, caffè): ISTAT li pubblica solo con il dato definitivo.

In [1002]:
print(istat_new[istat_new[COL_CODICE].isin(PRODOTTI.keys())].groupby(COL_CODICE).size())

new_largo = istat_largo(istat_new)
new_largo

ECOICOP_2
011131     9
01115     10
01141     10
011511     9
01181     10
012201     9
0722      10
dtype: int64


,pane,pasta,latte,olio_girasole,zucchero,caffe,carburanti
data,,,,,,,
2026-01-01,100.9,100.0,99.9,100.6,100.1,105.6,97.5
2026-02-01,100.8,99.2,100.1,100.1,99.6,104.6,98.6
2026-03-01,101.1,99.3,99.5,100.5,99.0,105.1,106.3
2026-04-01,101.0,99.8,99.3,101.0,98.9,105.4,112.3
2026-05-01,101.3,99.3,99.0,102.9,98.5,104.8,113.6
2026-06-01,101.3,99.6,98.6,103.1,98.3,104.6,113.0
2026-07-01,101.5,99.4,98.4,103.3,97.2,105.0,112.7
2026-08-01,101.6,100.1,98.6,103.1,97.7,105.3,118.9
2026-09-01,101.8,97.7,98.7,101.9,97.4,104.1,125.0


In [1003]:
coeff_file = RAW / "istat_coefficienti_raccordo.xlsx"
xls = pd.ExcelFile(coeff_file)
print(xls.sheet_names)

['Indice', 'IT', 'EN']


C:\Users\emanuele\AppData\Local\Temp\ipykernel_40600\2142935583.py:2: Pandas4Warning: The default engine for reading 'xlsx' files will change from 'openpyxl' to 'calamine' in a future version. Pass engine='openpyxl' (or set the 'io.excel.xlsx.reader' option) to keep the current engine and silence this warning.
  xls = pd.ExcelFile(coeff_file)


In [1004]:
coeff = pd.read_excel(coeff_file, sheet_name="IT")
coeff.head(15)

C:\Users\emanuele\AppData\Local\Temp\ipykernel_40600\3405767908.py:1: Pandas4Warning: The default engine for reading 'xlsx' files will change from 'openpyxl' to 'calamine' in a future version. Pass engine='openpyxl' (or set the 'io.excel.xlsx.reader' option) to keep the current engine and silence this warning.
  coeff = pd.read_excel(coeff_file, sheet_name="IT")


,Indice dei prezzi al consumo per l'intera collettività (NIC) per livelli EcoicopV2 - Anno 2026,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5
0,Tabella dei coefficienti di raccordo,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN


In [1005]:
coeff = pd.read_excel(
    coeff_file,
    sheet_name="IT",
    header=14,                      # la riga con i veri nomi delle colonne
    dtype={"ECOICOPv2": str},       # i codici come testo, come in ISTAT
)
print(coeff.columns.tolist())
coeff.head()

C:\Users\emanuele\AppData\Local\Temp\ipykernel_40600\2576227149.py:1: Pandas4Warning: The default engine for reading 'xlsx' files will change from 'openpyxl' to 'calamine' in a future version. Pass engine='openpyxl' (or set the 'io.excel.xlsx.reader' option) to keep the current engine and silence this warning.
  coeff = pd.read_excel(


['ECOICOPv2', 'livello', 'denominazioni ', 'coefficiente di raccordo da base 1995 a base 2010 ', 'coefficiente di raccordo da base 2010 a base 2015', 'coefficiente di raccordo da base 2015 a base 2025']


,ECOICOPv2,livello,denominazioni,coefficiente di raccordo da base 1995 a base 2010,coefficiente di raccordo da base 2010 a base 2015,coefficiente di raccordo da base 2015 a base 2025
0,00,Gen.,Indice generale,1.398,1.075,1.226
1,00st,Gen.ST,Indice generale senza tabacchi,1.39,1.073,"1,228 (r)"
2,01,Div.,Prodotti alimentari e bevande analcoliche,1.375,1.088,1.344
3,01.1,Grp.,Prodotti Alimentari,1.386,1.089,1.345
4,01.1.1,Clas.,Cereali e prodotti a base di cereali,1.386,"1,072 (r)",1.296


In [1006]:
#------------------------PULIZIA------------------------
coeff.columns = coeff.columns.str.strip()             # toglie gli spazi dai nomi delle colonne
COL_COEFF = coeff.columns[5]                          # colonna "da base 2015 a base 2025"
coeff["codice"] = coeff["ECOICOPv2"].str.replace(".", "", regex=False)  # 01.1.1 → 0111
coeff["coeff_2015_2025"] = pd.to_numeric(
    coeff[COL_COEFF].astype(str)
        .str.replace(" (r)", "", regex=False)         # toglie la nota "rivisto"
        .str.replace(",", ".", regex=False),          # virgola → punto
    errors="coerce",
)
#------------------------SELEZIONE----------------------
coeff_sel = coeff[coeff["codice"].isin(PRODOTTI.keys())]
coeff_sel[["codice", "denominazioni", "coeff_2015_2025"]]

,codice,denominazioni,coeff_2015_2025
14,011131,Pane,1.315
18,01115,"Maccheroni, tagliatelle, cuscus e paste alimen...",1.313
53,01141,Latte crudo e intero,1.255
72,011511,Olio di Semi Girasole,1.401
150,01181,Zucchero di canna e zucchero di barbabietola,1.314
188,012201,Caffè,1.422
467,0722,Carburanti e lubrificanti per mezzi personali ...,1.183


In [1007]:
coeff_serie = coeff_sel.set_index("codice")["coeff_2015_2025"].rename(index=PRODOTTI)   #["coeff_2015_2025"]-->prende solo questa colonna
coeff_serie

codice
pane             1.315
pasta            1.313
latte            1.255
olio_girasole    1.401
zucchero         1.314
caffe            1.422
carburanti       1.183
Name: coeff_2015_2025, dtype: float64

In [1008]:
#Voglio trovare i prezzi dal 2015 al 2025 mettendoli in base allo standard 2025, quindi devo dividere.
old_2025 = old_largo / coeff_serie   #pandas fa allineamento: abbina i dati per nome, non per posizione. 

In [1009]:
#VERIFICA: la media del 2025 dovrebbe venire circa 100
old_2025.loc["2025"].mean().round(1)

pane             100.0
pasta            100.0
latte            100.0
olio_girasole    100.0
zucchero         100.0
caffe            100.0
carburanti       100.0
dtype: float64

### Come leggere i valori della tabella UNICA

I valori **non sono prezzi in euro**: sono gli **indici NIC** di ciascun prodotto,
in **base 2025 = 100** (100 è il prezzo medio del prodotto nel 2025).

**Valore − 100 = variazione percentuale rispetto alla media del 2025**

| Esempio | Valore | Significato |
|---|---|---|
| Carburanti, settembre 2026 | 125,0 | costavano il 25% in più rispetto al 2025 |
| Caffè, gennaio 2026 | 105,6 | costava il 5,6% in più rispetto al 2025 |
| Pasta, dicembre 2025 | 97,7 | costava il 2,3% in meno rispetto alla media del 2025 |

Le serie 2016–2025, pubblicate da ISTAT in base 2015, sono state riportate in
base 2025 dividendo per i **coefficienti di raccordo ufficiali ISTAT**
(file `istat_coefficienti_raccordo.xlsx`, foglio `IT`).

In [1010]:
istat = pd.concat([old_2025, new_largo]).sort_index()
istat = istat.round(1)
istat.loc["2025-10":"2026-03"]  #.loc[righe,colonne]-->serve a scegliere una parte della tabella


,pane,pasta,latte,olio_girasole,zucchero,caffe,carburanti
data,,,,,,,
2025-10-01,100.6,99.3,100.4,99.6,100.6,104.7,98.2
2025-11-01,100.8,98.6,100.4,99.6,100.3,104.1,99.7
2025-12-01,100.7,97.7,100.5,100.0,98.8,104.5,99.5
2026-01-01,100.9,100.0,99.9,100.6,100.1,105.6,97.5
2026-02-01,100.8,99.2,100.1,100.1,99.6,104.6,98.6
2026-03-01,101.1,99.3,99.5,100.5,99.0,105.1,106.3


Nuovo dataset creato

In [1011]:
istat.to_csv(PROCESSED / "istat_base2025.csv")

# Inserisco il dataset FAO


In [1012]:
fao = pd.read_csv(
    RAW / "fao_food_price_index.csv",
    header=2,            # la riga con Date, Food Price Index, Meat...
    usecols=range(7),    # solo le prime 7 colonne, le altre sono vuote
)
fao = fao.dropna(how="all")   # toglie la riga vuota
fao.head()

,Date,Food Price Index,Meat,Dairy,Cereals,Oils,Sugar
1,1990-01,64.4,74.3,53.5,64.1,44.59,87.9
2,1990-02,64.7,76.8,52.2,62.2,44.50,90.7
3,1990-03,64.0,78.5,41.4,61.3,45.75,95.1
4,1990-04,66.0,81.2,48.4,62.8,44.02,94.3
5,1990-05,64.6,81.8,39.2,62.0,45.50,90.4


In [1013]:
#PULIZIA
fao = fao.rename(columns={
    "Date": "data",
    "Food Price Index": "fao_food",
    "Meat": "fao_carne",
    "Dairy": "fao_latticini",
    "Cereals": "fao_cereali",
    "Oils": "fao_oli",
    "Sugar": "fao_zucchero",
})
fao["data"] = pd.to_datetime(fao["data"])
fao = fao.set_index("data")
fao = fao.loc[INIZIO:]          # dal 2016, come i dati ISTAT
fao.head()


,fao_food,fao_carne,fao_latticini,fao_cereali,fao_oli,fao_zucchero
data,,,,,,
2016-01-01,84.9,84.4,79.4,87.6,85.29,86.9
2016-02-01,86.1,86.0,78.8,87.3,92.15,81.6
2016-03-01,87.4,86.1,76.7,86.5,97.34,95.5
2016-04-01,89.1,87.7,77.1,88.4,101.17,93.9
2016-05-01,90.6,90.2,72.7,91.1,99.56,104.8


### L’indice FAO, pubblicato in base 2014–2016 = 100, è stato riportato in base 2025 = 100 dividendo per la media del 2025.

In [1014]:
fao = fao / fao.loc["2025"].mean() * 100
fao = fao.round(1)
fao.loc["2025"].mean().round(1)   # controllo: devono venire tutti 100.0

fao_food         100.0
fao_carne        100.0
fao_latticini    100.0
fao_cereali      100.0
fao_oli          100.0
fao_zucchero     100.0
dtype: float64

In [1015]:
fao.to_csv(PROCESSED / "fao_base2025.csv") #aggiunge questo dataset in PROCESSED

## Passiamo al Brent, dal Pink Sheet della Banca Mondiale

In [1016]:
pink_file = RAW / "worldbank_pinksheet.xlsx"
print(pd.ExcelFile(pink_file).sheet_names)

['Mismatch Details', 'Monthly Prices', 'Monthly Indices', 'Description', 'Index Weights']


C:\Users\emanuele\AppData\Local\Temp\ipykernel_40600\4147738123.py:2: Pandas4Warning: The default engine for reading 'xlsx' files will change from 'openpyxl' to 'calamine' in a future version. Pass engine='openpyxl' (or set the 'io.excel.xlsx.reader' option) to keep the current engine and silence this warning.
  print(pd.ExcelFile(pink_file).sheet_names)


In [1017]:
pink_raw = pd.read_excel(pink_file, sheet_name="Monthly Prices", header=None)
pink_raw.iloc[:10, :8]  #.iloc[]-->il foglio ha decine di materie prime e non serve vederle tutte

C:\Users\emanuele\AppData\Local\Temp\ipykernel_40600\1188280447.py:1: Pandas4Warning: The default engine for reading 'xlsx' files will change from 'openpyxl' to 'calamine' in a future version. Pass engine='openpyxl' (or set the 'io.excel.xlsx.reader' option) to keep the current engine and silence this warning.
  pink_raw = pd.read_excel(pink_file, sheet_name="Monthly Prices", header=None)


,0,1,2,3,4,5,6,7
0,World Bank Commodity Price Data (The Pink Sheet),NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"monthly prices in nominal US dollars, 1960 to ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,(monthly series are available only in nominal ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Updated on November 03, 2026",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,"Crude oil, average","Crude oil, Brent","Crude oil, Dubai","Crude oil, WTI","Coal, Australian","Coal, South African **","Natural gas, US"
5,NaN,($/bbl),($/bbl),($/bbl),($/bbl),($/mt),($/mt),($/mmbtu)
6,1960M01,1.6,1.6,1.6,…,…,…,0.14
7,1960M02,1.6,1.6,1.6,…,…,…,0.14
8,1960M03,1.6,1.6,1.6,…,…,…,0.14
9,1960M04,1.6,1.6,1.6,…,…,…,0.14


In [1018]:
pink = pd.read_excel(pink_file, sheet_name="Monthly Prices", header=4)
pink = pink.iloc[1:]                                   # toglie la riga delle unità di misura
pink = pink.rename(columns={pink.columns[0]: "data"})  # dà un nome alla colonna delle date

parole = ["brent", "natural gas", "wheat", "sunflower", "coffee", "sugar", "urea"]
for p in parole:
    print(p, "->", [c for c in pink.columns if p in str(c).lower()])

C:\Users\emanuele\AppData\Local\Temp\ipykernel_40600\1246227489.py:1: Pandas4Warning: The default engine for reading 'xlsx' files will change from 'openpyxl' to 'calamine' in a future version. Pass engine='openpyxl' (or set the 'io.excel.xlsx.reader' option) to keep the current engine and silence this warning.
  pink = pd.read_excel(pink_file, sheet_name="Monthly Prices", header=4)


brent -> ['Crude oil, Brent']
natural gas -> ['Natural gas, US', 'Natural gas, Europe', 'Liquefied natural gas, Japan', 'Natural gas index']
wheat -> ['Wheat, US SRW', 'Wheat, US HRW']
sunflower -> ['Sunflower oil']
coffee -> ['Coffee, Arabica', 'Coffee, Robusta']
sugar -> ['Sugar, EU', 'Sugar, US', 'Sugar, world']
urea -> ['Urea ']


### Prezzi delle materie prime (World Bank Pink Sheet, dollari USA nominali)
| Colonna | Unità |
|---|---|
| brent | $/barile |
| gas_europa | $/mmbtu (milioni di BTU) |
| grano, girasole_mondo, urea | $/tonnellata |
| caffe_arabica, caffe_robusta, zucchero_mondo | $/kg |

In [1019]:
#PULIZIA
pink.columns = pink.columns.astype(str).str.strip()   # toglie gli spazi, come "Urea "

PINK = {
    "Crude oil, Brent": "brent",
    "Natural gas, Europe": "gas_europa",
    "Wheat, US HRW": "grano",
    "Sunflower oil": "girasole_mondo",
    "Coffee, Arabica": "caffe_arabica",
    "Coffee, Robusta": "caffe_robusta",
    "Sugar, world": "zucchero_mondo",
    "Urea": "urea",
}

pink = pink[["data"] + list(PINK.keys())].rename(columns=PINK)

# "1960M01" -> "1960-01" -> data vera
pink["data"] = pd.to_datetime(pink["data"].astype(str).str.replace("M", "-"), format="%Y-%m")
pink = pink.set_index("data")

# i "…" diventano NaN, il resto numeri
pink = pink.apply(pd.to_numeric, errors="coerce")

pink = pink.loc[INIZIO:]
pink.tail()

,brent,gas_europa,grano,girasole_mondo,caffe_arabica,caffe_robusta,zucchero_mondo,urea
data,,,,,,,,
2026-06-01,85.4,15.17,286.0,1508.0,6.79,3.73,0.32,453.1
2026-07-01,83.4,18.06,310.0,1497.0,7.91,4.07,0.34,400.0
2026-08-01,90.9,21.11,330.0,1484.0,7.97,3.98,0.38,390.0
2026-09-01,116.8,25.42,347.0,1393.0,7.26,3.69,0.41,407.5
2026-10-01,125.0,28.50,317.0,1413.0,7.40,3.63,0.43,420.0


In [1020]:
pink.to_csv(PROCESSED / "pinksheet.csv")    #aggiungo questo dataset in processed

## Passiamo al file EURO-DOLLARO

In [1021]:
bce_raw = pd.read_csv(RAW / "bce_eur_usd.csv")
print(bce_raw.shape)
print(bce_raw.columns.tolist())
bce_raw.head()

(334, 3)
['DATE', 'TIME PERIOD', 'US dollar/Euro ECB reference exchange rate (EXR.M.USD.EUR.SP00.A)']


,DATE,TIME PERIOD,US dollar/Euro ECB reference exchange rate (EXR.M.USD.EUR.SP00.A)
0,1999-01-31,1999Jan,1.1608
1,1999-02-28,1999Feb,1.1208
2,1999-03-31,1999Mar,1.0883
3,1999-04-30,1999Apr,1.0704
4,1999-05-31,1999May,1.0628


In [1022]:
bce = bce_raw.rename(columns={
    "DATE": "data",
    bce_raw.columns[2]: "eur_usd",       # il nome lunghissimo della serie
})[["data", "eur_usd"]]

bce["data"] = pd.to_datetime(bce["data"]).dt.to_period("M").dt.to_timestamp()  #Non sto falsificando i dati, è una media mensile, la data è una convenzione. Basta che sia il mese corretto
bce = bce.set_index("data").loc[INIZIO:]
bce.tail()

,eur_usd
data,
2026-06-01,1.1518
2026-07-01,1.1417
2026-08-01,1.1593
2026-09-01,1.1513
2026-10-01,1.1202


In [1023]:
bce.to_csv(PROCESSED / "bce_eur_usd.csv")   #aggiungo questo dataset in processed

# Passiamo al File Geopolitical Risk Index

In [1024]:
#LETTURA GPR (Geopolitical Risk Index, Caldara & Iacoviello)
gpr_file = RAW / "gpr_index.xls"

print(pd.ExcelFile(gpr_file, engine="calamine").sheet_names)   # i fogli del file

gpr_raw = pd.read_excel(gpr_file, engine="calamine")
print(gpr_raw.shape)              # (righe, colonne)
print(gpr_raw.columns.tolist())   # i nomi delle colonne
gpr_raw.head()

['Sheet1']
(1522, 115)
['month', 'GPR', 'GPRT', 'GPRA', 'GPRH', 'GPRHT', 'GPRHA', 'SHARE_GPR', 'N10', 'SHARE_GPRH', 'N3H', 'GPRH_NOEW', 'GPR_NOEW', 'GPRH_AND', 'GPR_AND', 'GPRH_BASIC', 'GPR_BASIC', 'SHAREH_CAT_1', 'SHAREH_CAT_2', 'SHAREH_CAT_3', 'SHAREH_CAT_4', 'SHAREH_CAT_5', 'SHAREH_CAT_6', 'SHAREH_CAT_7', 'SHAREH_CAT_8', 'GPRC_ARG', 'GPRC_AUS', 'GPRC_BEL', 'GPRC_BRA', 'GPRC_CAN', 'GPRC_CHE', 'GPRC_CHL', 'GPRC_CHN', 'GPRC_COL', 'GPRC_DEU', 'GPRC_DNK', 'GPRC_EGY', 'GPRC_ESP', 'GPRC_FIN', 'GPRC_FRA', 'GPRC_GBR', 'GPRC_HKG', 'GPRC_HUN', 'GPRC_IDN', 'GPRC_IND', 'GPRC_ISR', 'GPRC_ITA', 'GPRC_JPN', 'GPRC_KOR', 'GPRC_MEX', 'GPRC_MYS', 'GPRC_NLD', 'GPRC_NOR', 'GPRC_PER', 'GPRC_PHL', 'GPRC_POL', 'GPRC_PRT', 'GPRC_RUS', 'GPRC_SAU', 'GPRC_SWE', 'GPRC_THA', 'GPRC_TUN', 'GPRC_TUR', 'GPRC_TWN', 'GPRC_UKR', 'GPRC_USA', 'GPRC_VEN', 'GPRC_VNM', 'GPRC_ZAF', 'GPRHC_ARG', 'GPRHC_AUS', 'GPRHC_BEL', 'GPRHC_BRA', 'GPRHC_CAN', 'GPRHC_CHE', 'GPRHC_CHL', 'GPRHC_CHN', 'GPRHC_COL', 'GPRHC_DEU', 'GPRHC_DNK', 'GP

,month,GPR,GPRT,GPRA,GPRH,GPRHT,GPRHA,SHARE_GPR,N10,SHARE_GPRH,...,GPRHC_TUN,GPRHC_TUR,GPRHC_TWN,GPRHC_UKR,GPRHC_USA,GPRHC_VEN,GPRHC_VNM,GPRHC_ZAF,var_name,var_label
0,1900-01-01,NaN,NaN,NaN,87.927849,64.717491,110.453522,NaN,NaN,3.171932,...,0.0,0.038840,0.0,0.0,2.718799,0.051787,0.012947,1.152253,month,Date (year/month)
1,1900-02-01,NaN,NaN,NaN,86.566490,71.936844,96.250488,NaN,NaN,3.122822,...,0.0,0.125471,0.0,0.0,2.732469,0.027882,0.000000,1.143176,GPR,Recent GPR (Index: 1985:2019=100)
2,1900-03-01,NaN,NaN,NaN,72.140701,57.475853,84.499428,NaN,NaN,2.602422,...,0.0,0.180366,0.0,0.0,2.151507,0.025767,0.000000,0.863180,GPRT,Recent GPR Threats (Index: 1985:2019=100)
3,1900-04-01,NaN,NaN,NaN,54.419449,37.326603,65.858208,NaN,NaN,1.963141,...,0.0,0.066774,0.0,0.0,1.776175,0.000000,0.000000,0.641026,GPRA,Recent GPR Acts (Index: 1985:2019=100)
4,1900-05-01,NaN,NaN,NaN,64.405197,48.200008,74.373955,NaN,NaN,2.323370,...,0.0,0.081522,0.0,0.0,1.970109,0.013587,0.000000,0.788043,GPRH,Historical GPR (Index: 1900:2019=100)


## Geopolitical Risk Index (Caldara & Iacoviello)

Fonte: [matteoiacoviello.com/gpr.htm](https://www.matteoiacoviello.com/gpr.htm) – file `gpr_index.xls`.
L'indice si costruisce contando gli articoli di 10 grandi giornali americani e britannici
che parlano di guerre, minacce, attacchi e tensioni internazionali.

### Colonne selezionate

| Colonna | Serie originale | Cosa misura | Eventi collegati |
|---|---|---|---|
| `gpr` | GPR | Rischio geopolitico globale | tutti |
| `gpr_minacce` | GPRT | Solo minacce e tensioni (*Threats*) | fasi precedenti alle guerre |
| `gpr_atti` | GPRA | Solo atti concreti: guerre, attacchi (*Acts*) | inizio dei conflitti |
| `gpr_russia` | GPRC_RUS | Rischio che coinvolge la Russia | Crimea 2014, Ucraina 2022, grano 2023 |
| `gpr_ucraina` | GPRC_UKR | Rischio che coinvolge l'Ucraina | Ucraina 2022, grano 2023 |
| `gpr_israele` | GPRC_ISR | Rischio che coinvolge Israele | Gaza 2023, Iran 2026 |
| `gpr_arabia` | GPRC_SAU | Rischio che coinvolge l'Arabia Saudita | Golfo Persico, Iran 2026 |
| `gpr_egitto` | GPRC_EGY | Rischio che coinvolge l'Egitto | Suez e Mar Rosso 2023 |
| `gpr_usa` | GPRC_USA | Rischio che coinvolge gli USA | dazi 2025, Iran 2026 |
| `gpr_cina` | GPRC_CHN | Rischio che coinvolge la Cina | dazi 2025 |
| `gpr_italia` | GPRC_ITA | Rischio che coinvolge l'Italia | confronto |

### Come leggere i valori

Le colonne hanno **due scale diverse**, da non confrontare tra loro:

- **`gpr`, `gpr_minacce`, `gpr_atti`** sono **indici** in base 1985–2019 = 100.
  Esempio: `gpr` = 150 → rischio geopolitico del 50% sopra la media storica.
- **Le colonne per paese** sono **percentuali**: la quota di articoli del mese
  che collegano quel paese a guerre, minacce o attacchi.
  Esempio: `gpr_usa` = 3,76 → il 3,76% degli articoli.

Gli indici per paese **non distinguono** se il paese è aggressore, bersaglio o luogo
dell'evento: misurano quanto il paese è al centro delle notizie di tensione.

### Limiti
- **Manca l'Iran**: l'indice copre 44 paesi, ma non l'Iran. La crisi del 2026 è
  catturata indirettamente da `gpr_israele`, `gpr_arabia` e `gpr_usa`.
- **Punto di vista occidentale**: i giornali sono americani e britannici, quindi
  l'indice misura il rischio come viene raccontato in Occidente.
  
La base originale è stata mantenuta: il GPR non è un prezzo, e il cambio di base
  non modificherebbe le variazioni.

In [1025]:
GPR = {
    "GPR": "gpr", "GPRT": "gpr_minacce", "GPRA": "gpr_atti",
    "GPRC_RUS": "gpr_russia", "GPRC_UKR": "gpr_ucraina",
    "GPRC_ISR": "gpr_israele", "GPRC_SAU": "gpr_arabia",
    "GPRC_EGY": "gpr_egitto", "GPRC_USA": "gpr_usa",
    "GPRC_CHN": "gpr_cina", "GPRC_ITA": "gpr_italia",
}

gpr = gpr_raw[["month"] + list(GPR.keys())].rename(columns={"month": "data", **GPR})
gpr["data"] = pd.to_datetime(gpr["data"])
gpr = gpr.set_index("data").loc[INIZIO:]
gpr = gpr.dropna(how="all")      # toglie eventuali mesi futuri ancora vuoti
gpr.tail()

,gpr,gpr_minacce,gpr_atti,gpr_russia,gpr_ucraina,gpr_israele,gpr_arabia,gpr_egitto,gpr_usa,gpr_cina,gpr_italia
data,,,,,,,,,,,
2026-06-01,177.999832,212.192490,184.130585,1.302018,1.090207,2.498131,0.448542,0.186893,4.597558,0.847246,0.242960
2026-07-01,168.686050,192.278778,181.616669,1.518493,1.190851,1.902842,1.146746,0.359146,4.259341,1.153046,0.126016
2026-08-01,125.745857,139.014862,127.285515,1.228727,0.952756,1.163020,0.617649,0.210263,3.232801,1.025035,0.177410
2026-09-01,155.939758,181.259201,154.392746,1.552635,1.166087,1.610617,0.979255,0.322123,3.975003,1.249839,0.186832
2026-10-01,150.520004,156.843567,167.572296,1.723784,1.498050,1.272317,0.855052,0.218893,3.762227,0.841371,0.191532


In [1026]:
gpr = gpr.round(2)
gpr.loc["2026-01":"2026-05", ["gpr", "gpr_atti", "gpr_usa", "gpr_israele", "gpr_arabia"]]#-->per vedere se dopo l'attacco all'iran i valori sono cresciuti

,gpr,gpr_atti,gpr_usa,gpr_israele,gpr_arabia
data,,,,,
2026-01-01,168.20,100.22,4.55,1.16,0.34
2026-02-01,121.62,83.00,3.12,1.13,0.32
2026-03-01,330.18,450.36,8.68,6.39,2.24
2026-04-01,253.41,310.52,6.71,3.82,0.90
2026-05-01,203.78,213.58,5.31,2.23,0.59


In [1027]:
gpr.to_csv(PROCESSED / "gpr.csv") #aggiunto in processed